# DeepSearch — Colab GPU Backend

This notebook turns your Colab GPU into a **free, unlimited LLM inference server** for DeepSearch.

### How it works
1. **Cell 1** — Installs vLLM (the inference engine)
2. **Cell 2** — Creates a public tunnel via ngrok
3. **Cell 3** — Downloads the model and starts the server

### Prerequisites
- Set runtime to **GPU**: `Runtime > Change runtime type > T4 GPU`
- Get a free ngrok token: https://dashboard.ngrok.com/authtokens

---

In [ ]:
# ============================================
# CELL 1: Install Dependencies (~3-5 min)
# ============================================

!pip install vllm -q
!pip install pyngrok -q

# Verify GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ============================================
# CELL 2: Configure ngrok tunnel
# ============================================

from pyngrok import ngrok

# PASTE YOUR NGROK AUTH TOKEN BELOW
# Get it free: https://dashboard.ngrok.com/authtokens
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- REPLACE THIS

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Create tunnel to port 8000
public_url = ngrok.connect(8000, "http").public_url

print()
print("=" * 60)
print("YOUR DEEPSEARCH BACKEND URL:")
print(f"   {public_url}/v1")
print("=" * 60)
print()
print("Copy this into your local DeepSearch/.env file:")
print(f"  LLM_BASE_URL={public_url}/v1")
print(f"  LLM_API_KEY=not-needed")
print(f"  LLM_BACKEND=colab")
print()
print("Now run Cell 3 to start the model server...")

In [ ]:
# ============================================
# CELL 3: Start vLLM Server
# This cell will KEEP RUNNING - that means
# the server is alive. Do NOT stop it.
# ============================================

# MODEL SELECTION
# Lightweight (T4, 15GB VRAM):
#   Qwen/Qwen2.5-3B-Instruct       ~6 GB, fast
#   Qwen/Qwen2.5-7B-Instruct-AWQ   ~4 GB, quantized
# Medium (L4 / A100):
#   Qwen/Qwen2.5-14B-Instruct-AWQ  ~8 GB
#   Qwen/Qwen2.5-32B-Instruct-AWQ  ~18 GB
# Full Power (A100 80GB):
#   Qwen/Qwen2.5-72B-Instruct-AWQ  ~40 GB

MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_MODEL_LEN = 16384  # 16k context, safe for T4

print(f"Loading: {MODEL}")
print(f"Context: {MAX_MODEL_LEN} tokens")
print(f"Downloading weights (~3-8 min first time)...")
print()

!python -m vllm.entrypoints.openai.api_server \
    --model $MODEL \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len $MAX_MODEL_LEN \
    --dtype auto \
    --trust-remote-code \
    --enforce-eager

In [ ]:
# ============================================
# CELL 4 (OPTIONAL): Health Check
# Run this in a NEW cell while Cell 3 runs
# ============================================

import requests

# public_url variable is still in memory from Cell 2
BASE = f"{public_url}/v1"

# List models
r = requests.get(f"{BASE}/models")
print("Available models:", r.json())

# Quick test
r = requests.post(f"{BASE}/chat/completions", json={
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "messages": [{"role": "user", "content": "Say hello in one word."}],
    "max_tokens": 10,
    "temperature": 0.0
})
print("Response:", r.json()["choices"][0]["message"]["content"])